# BudgetMem Final Polish - Error Bars + Budget Sensitivity

Two experiments to finalize the paper:
1. **3-seed medium-doc runs** (~3-4 hours) - adds error bars to headline result
2. **Budget sensitivity re-run** (~3 hours) - fixes number inconsistency

Run all cells top to bottom. Estimated total: ~6-7 hours.

In [ ]:
!pip install -q transformers accelerate datasets rank-bm25 nltk scikit-learn tqdm

import torch
import numpy as np
import json
import re
import string
import time
import random
import os
from datetime import datetime
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

PROJECT_DIR = "/content/budgetmem_final_polish"
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("Setup done.")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
model.eval()
print("Model loaded.")

## Shared code (same as revision notebook)

In [ ]:
# ── F1 scoring ──
def normalize_answer(s):
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    s = ' '.join(s.split())
    return s

def compute_f1(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()
    if not pred_tokens or not truth_tokens:
        return float(pred_tokens == truth_tokens)
    common = set(pred_tokens) & set(truth_tokens)
    if not common:
        return 0.0
    prec = len(common) / len(pred_tokens)
    rec  = len(common) / len(truth_tokens)
    return 2 * prec * rec / (prec + rec)

# ── Answer generation ──
def generate_answer(context, question, max_ctx_chars=8000):
    prompt = f"""Context: {context[:max_ctx_chars]}

Question: {question}

Answer the question based only on the context above. Be concise.

Answer:"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=100, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    answer = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return answer.strip()

# ── Chunking ──
def chunk_document(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = ' '.join(words[i:i + chunk_size])
        if len(chunk.split()) > 10:
            chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

# ── BM25 retrieval ──
def retrieve_bm25(query, chunks, top_k=3):
    if not chunks:
        return []
    tokenized = [c.split() for c in chunks]
    bm25 = BM25Okapi(tokenized)
    scores = bm25.get_scores(query.split())
    k = min(top_k, len(chunks))
    top_idx = np.argsort(scores)[-k:][::-1]
    return [chunks[i] for i in top_idx]

print("Utilities loaded.")

In [ ]:
# ── BudgetMem salience scorer (6 features, paper weights) ──

DISCOURSE_MARKERS = [
    'however', 'therefore', 'moreover', 'furthermore', 'consequently',
    'nevertheless', 'additionally', 'specifically', 'importantly',
    'in conclusion', 'on the other hand', 'as a result', 'for example',
    'in contrast', 'meanwhile', 'subsequently', 'nonetheless',
    'accordingly', 'hence', 'thus', 'indeed', 'notably',
    'in particular', 'conversely', 'alternatively', 'likewise',
    'similarly', 'in summary', 'to summarize', 'overall',
    'in other words', 'that is', 'namely', 'first', 'second',
    'third', 'finally', 'next', 'then', 'afterward',
    'before', 'after', 'during', 'while', 'although',
    'despite', 'regardless', 'provided that'
]

DEFAULT_WEIGHTS = {
    'entity_density':   0.20,
    'tfidf_importance': 0.20,
    'position_bias':    0.15,
    'numerical_density': 0.15,
    'discourse_markers': 0.10,
    'question_presence': 0.10,
}

def compute_features(chunks):
    n = len(chunks)
    if n == 0:
        return {}

    entity_raw = np.zeros(n)
    for i, c in enumerate(chunks):
        words = c.split()
        if words:
            entity_raw[i] = sum(1 for w in words if w and w[0].isupper()) / len(words)

    tfidf_raw = np.zeros(n)
    try:
        vec = TfidfVectorizer(max_features=500, stop_words='english')
        mat = vec.fit_transform(chunks)
        tfidf_raw = np.array(mat.mean(axis=1)).flatten()
    except Exception:
        pass

    position_raw = np.zeros(n)
    for i in range(n):
        position_raw[i] = min(1.0, max(0.0, 1.3 - 2.0 * abs(i / n - 0.5)))

    number_raw = np.zeros(n)
    for i, c in enumerate(chunks):
        words = c.split()
        if words:
            nums = sum(1 for w in words if re.search(r'\d', w))
            number_raw[i] = nums / len(words)

    discourse_raw = np.zeros(n)
    for i, c in enumerate(chunks):
        lower_c = c.lower()
        words = c.split()
        count = sum(1 for m in DISCOURSE_MARKERS if m in lower_c)
        discourse_raw[i] = count / max(len(words), 1)

    question_raw = np.zeros(n)
    interrogatives = {'who', 'what', 'where', 'when', 'why', 'how'}
    for i, c in enumerate(chunks):
        has_qmark = '?' in c
        has_interrog = bool(set(c.lower().split()) & interrogatives)
        q_count = int(has_qmark) + int(has_interrog)
        question_raw[i] = min(1.0, q_count / 2.0)

    def norm(arr):
        mx = arr.max()
        if mx > 0:
            return arr / mx
        return arr

    return {
        'entity_density':    norm(entity_raw),
        'tfidf_importance':  norm(tfidf_raw),
        'position_bias':     norm(position_raw),
        'numerical_density': norm(number_raw),
        'discourse_markers': norm(discourse_raw),
        'question_presence': norm(question_raw),
    }

def compute_salience(features_dict, weights=None):
    if weights is None:
        weights = DEFAULT_WEIGHTS
    n = len(next(iter(features_dict.values())))
    scores = np.zeros(n)
    for feat_name, feat_arr in features_dict.items():
        w = weights.get(feat_name, 0)
        scores += w * feat_arr
    return scores

def budgetmem_answer(document, question, budget_ratio=0.3, top_k=3):
    chunks = chunk_document(document)
    if not chunks:
        return generate_answer('', question)
    features = compute_features(chunks)
    salience = compute_salience(features)
    k = max(1, int(len(chunks) * budget_ratio))
    top_indices = np.argsort(salience)[-k:][::-1]
    selected = [chunks[i] for i in sorted(top_indices)]
    retrieved = retrieve_bm25(question, selected, top_k=top_k)
    context = '\n\n'.join(retrieved)
    return generate_answer(context, question)

def baseline_answer(document, question, top_k=3):
    chunks = chunk_document(document)
    if not chunks:
        return generate_answer('', question)
    retrieved = retrieve_bm25(question, chunks, top_k=top_k)
    context = '\n\n'.join(retrieved)
    return generate_answer(context, question)

print("BudgetMem scorer loaded.")

## Generate medium-length documents

In [ ]:
def create_research_paper(idx):
    topics = [
        'machine learning', 'deep learning', 'neural networks',
        'natural language processing', 'computer vision',
        'reinforcement learning', 'transfer learning', 'attention mechanisms',
        'graph neural networks', 'federated learning',
    ]
    methods = [
        'transformer', 'convolutional network', 'recurrent network',
        'generative adversarial network', 'variational autoencoder',
        'BERT', 'GPT', 'ResNet', 'diffusion model', 'mixture of experts',
    ]
    datasets_list = [
        'ImageNet', 'CIFAR-10', 'GLUE', 'SQuAD', 'COCO',
        'WMT', 'CommonCrawl', 'WikiText-103', 'Penn Treebank', 'LibriSpeech',
    ]

    # Use idx-based seed so papers are deterministic per idx
    rng = random.Random(idx + 1000)
    topic  = topics[idx % len(topics)]
    method = methods[idx % len(methods)]
    ds     = datasets_list[idx % len(datasets_list)]
    acc    = round(85 + rng.uniform(0, 12), 1)
    prev   = round(acc - rng.uniform(2, 8), 1)
    lr     = round(rng.choice([1e-3, 3e-4, 5e-4, 1e-4]), 5)
    bs     = rng.choice([16, 32, 64, 128])
    epochs = rng.choice([50, 100, 200, 300])

    abstract = (
        f"This paper presents a novel approach to {topic} using a {method} architecture. "
        f"We evaluate on {ds} and achieve {acc}% accuracy, surpassing the previous "
        f"state-of-the-art result of {prev}%. Our method requires no task-specific "
        f"fine-tuning and trains in under 24 hours on a single GPU. "
        f"We release code and pretrained weights to support reproducibility."
    )
    intro = (
        f"The field of {topic} has advanced rapidly over the past decade. "
        f"Classical approaches relied on hand-crafted feature pipelines that "
        f"demanded substantial domain expertise. The rise of deep learning "
        f"replaced many of these pipelines with end-to-end trainable models, "
        f"but challenges in scalability, generalization, and data efficiency "
        f"persist. Recent work on {method} architectures has shown promise, "
        f"yet existing implementations often require prohibitive compute budgets. "
        f"In this work we propose a lightweight variant that retains the "
        f"representational power of {method} while cutting training cost by 60%. "
        f"Our key contributions are: (1) a parameter-efficient adaptation scheme, "
        f"(2) a curriculum-based training strategy, and (3) extensive evaluation "
        f"on {ds} showing {acc}% accuracy. "
    ) * 8
    related = (
        f"Prior work in {topic} spans several decades. Early statistical methods "
        f"achieved moderate success on constrained benchmarks but struggled with "
        f"real-world variability. The introduction of deep neural networks marked a "
        f"turning point, with convolutional and recurrent architectures dominating "
        f"leaderboards. More recently, attention-based models have set new records "
        f"across a range of tasks. However, these models are computationally "
        f"expensive and their environmental impact has drawn criticism. Several "
        f"groups have proposed efficiency improvements including pruning, "
        f"quantization, and knowledge distillation, but gains come at the cost of "
        f"accuracy. Our approach differs in that we modify the architecture itself "
        f"rather than applying post-hoc compression. "
    ) * 7
    methodology = (
        f"Our proposed {method} variant introduces three modifications to the "
        f"standard architecture. First, we replace dense layers with sparse "
        f"mixtures of experts, activating only 25% of parameters per forward pass. "
        f"Second, we apply rotary position embeddings to improve length "
        f"generalization. Third, we use a staged training curriculum that "
        f"begins with short sequences and gradually increases context length. "
        f"Training uses AdamW with a learning rate of {lr}, batch size {bs}, "
        f"for {epochs} epochs on 8 A100 GPUs. We apply gradient clipping at 1.0 "
        f"and use cosine learning rate decay with a 5% warmup period. "
        f"All hyperparameters were selected via grid search on a held-out "
        f"validation set of 5000 examples. "
    ) * 10
    results = (
        f"On {ds}, our model achieves {acc}% accuracy, compared to {prev}% "
        f"for the previous best method. The improvement is statistically "
        f"significant (p < 0.01, paired bootstrap test). Ablation studies confirm "
        f"that each of our three modifications contributes to the final result: "
        f"removing sparse experts drops accuracy to {round(acc-3.2,1)}%, removing "
        f"rotary embeddings to {round(acc-1.8,1)}%, and removing the curriculum to "
        f"{round(acc-2.5,1)}%. Training wall-clock time is 18 hours, versus 45 hours "
        f"for the dense baseline. Inference latency is 12ms per example on a "
        f"single V100 GPU. "
    ) * 7
    discussion = (
        f"Our results demonstrate that architectural efficiency and high accuracy "
        f"are not mutually exclusive in {topic}. The sparse mixture of experts "
        f"approach activates fewer parameters without sacrificing representational "
        f"capacity, and the training curriculum reduces wasted computation on "
        f"easy examples. However, limitations remain: our method has not been "
        f"tested on languages other than English, and the expert routing mechanism "
        f"adds implementation complexity. Future work will explore multilingual "
        f"settings and distillation of the sparse model into a smaller dense one. "
    ) * 5
    acknowledgements = (
        f"This work was supported by the National Science Foundation under grant "
        f"IIS-{rng.randint(1800000,2100000)}. We thank the anonymous reviewers "
        f"for their constructive feedback and our colleagues for helpful discussions. "
        f"Computational resources were provided by the university HPC cluster."
    )

    paper = (
        f"Title: A Novel Approach to {topic.title()} Using {method.title()}\n\n"
        f"Abstract: {abstract}\n\n"
        f"1. Introduction\n{intro}\n\n"
        f"2. Related Work\n{related}\n\n"
        f"3. Methodology\n{methodology}\n\n"
        f"4. Results\n{results}\n\n"
        f"5. Discussion\n{discussion}\n\n"
        f"Acknowledgements\n{acknowledgements}"
    )
    qa_pairs = [
        (f"What accuracy does the proposed method achieve on {ds}?", f"{acc}%"),
        (f"What was the previous state-of-the-art accuracy?", f"{prev}%"),
        (f"What learning rate was used for training?", f"{lr}"),
        (f"How many epochs was the model trained for?", f"{epochs}"),
        (f"What batch size was used during training?", f"{bs}"),
    ]
    return paper, qa_pairs

# Generate all papers and QA pairs (deterministic, seed-independent)
all_papers = []
all_qa = []
for idx in range(40):
    paper, qas = create_research_paper(idx)
    for q, a in qas:
        all_papers.append(paper)
        all_qa.append({'question': q, 'answer': a})

print(f"Generated {len(all_qa)} QA pairs from 40 papers.")
print(f"Avg paper length: {np.mean([len(p.split()) for p in all_papers[::5]]):.0f} tokens")

---
## Experiment 1: 3-Seed Medium-Doc Runs (Error Bars)

Run Baseline RAG and BudgetMem 3 times each with different seeds.
Seeds only affect the model's generation (via torch manual seed).
This gives us mean +/- std for the paper's Table 2.

In [ ]:
SEEDS = [42, 123, 456]

baseline_per_seed = []
budgetmem_per_seed = []

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"SEED {seed}")
    print(f"{'='*60}")

    torch.manual_seed(seed)
    np.random.seed(seed)

    # Baseline RAG
    base_f1s = []
    for i in tqdm(range(len(all_qa)), desc=f"Baseline seed={seed}"):
        pred = baseline_answer(all_papers[i], all_qa[i]['question'])
        f1 = compute_f1(pred, all_qa[i]['answer'])
        base_f1s.append(f1)
    base_mean = np.mean(base_f1s)
    baseline_per_seed.append(base_mean)
    print(f"  Baseline F1: {base_mean:.4f}")

    # BudgetMem
    torch.manual_seed(seed)
    bm_f1s = []
    for i in tqdm(range(len(all_qa)), desc=f"BudgetMem seed={seed}"):
        pred = budgetmem_answer(all_papers[i], all_qa[i]['question'])
        f1 = compute_f1(pred, all_qa[i]['answer'])
        bm_f1s.append(f1)
    bm_mean = np.mean(bm_f1s)
    budgetmem_per_seed.append(bm_mean)
    print(f"  BudgetMem F1: {bm_mean:.4f}")

    # Checkpoint after each seed
    checkpoint = {
        'completed_seeds': len(baseline_per_seed),
        'baseline_per_seed': baseline_per_seed,
        'budgetmem_per_seed': budgetmem_per_seed,
    }
    with open(f"{PROJECT_DIR}/results/3seed_checkpoint.json", 'w') as f:
        json.dump(checkpoint, f, indent=2)
    print(f"  Checkpoint saved ({len(baseline_per_seed)}/3 seeds done).")

print("\nAll 3 seeds complete!")

In [ ]:
# Print results for paper
base_mean = np.mean(baseline_per_seed)
base_std = np.std(baseline_per_seed)
bm_mean = np.mean(budgetmem_per_seed)
bm_std = np.std(budgetmem_per_seed)

print("="*60)
print("TABLE 2 UPDATE: Medium Documents (3 seeds)")
print("="*60)
print(f"Baseline RAG:  {base_mean:.4f} +/- {base_std:.4f}")
print(f"BudgetMem:     {bm_mean:.4f} +/- {bm_std:.4f}")
print(f"Per-seed baseline: {[f'{x:.4f}' for x in baseline_per_seed]}")
print(f"Per-seed budgetmem: {[f'{x:.4f}' for x in budgetmem_per_seed]}")

three_seed_results = {
    'experiment': '3-seed medium doc comparison',
    'seeds': SEEDS,
    'n_qa_pairs': len(all_qa),
    'baseline': {
        'mean': float(base_mean), 'std': float(base_std),
        'per_seed': [float(x) for x in baseline_per_seed]
    },
    'budgetmem': {
        'mean': float(bm_mean), 'std': float(bm_std),
        'per_seed': [float(x) for x in budgetmem_per_seed]
    },
    'timestamp': datetime.now().isoformat()
}
with open(f"{PROJECT_DIR}/results/three_seed_medium.json", 'w') as f:
    json.dump(three_seed_results, f, indent=2)
print("\nSaved to three_seed_medium.json")

---
## Experiment 2: Budget Sensitivity (7 ratios, consistent code)

Re-run the budget sweep with the same code that produced
the headline result, fixing the number inconsistency.

In [ ]:
BUDGET_RATIOS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.70, 0.90]

# Use first 100 QA pairs for budget sensitivity (faster, same as original)
n_budget = 100
budget_docs = all_papers[:n_budget]
budget_qa = all_qa[:n_budget]

budget_results = {}

# First run baseline (no compression) as the reference
torch.manual_seed(42)
base_f1s = []
for i in tqdm(range(n_budget), desc="Baseline (budget ref)"):
    pred = baseline_answer(budget_docs[i], budget_qa[i]['question'])
    f1 = compute_f1(pred, budget_qa[i]['answer'])
    base_f1s.append(f1)
budget_baseline_f1 = np.mean(base_f1s)
print(f"Baseline F1 (100 examples): {budget_baseline_f1:.4f}")

# Sweep budget ratios
for ratio in BUDGET_RATIOS:
    torch.manual_seed(42)
    f1_scores = []
    storage_saved = []

    for i in tqdm(range(n_budget), desc=f"Budget {int(ratio*100)}%"):
        chunks = chunk_document(budget_docs[i])
        if not chunks:
            f1_scores.append(0.0)
            continue

        features = compute_features(chunks)
        salience = compute_salience(features)
        k = max(1, int(len(chunks) * ratio))
        top_indices = np.argsort(salience)[-k:][::-1]
        selected = [chunks[j] for j in sorted(top_indices)]

        retrieved = retrieve_bm25(budget_qa[i]['question'], selected, top_k=3)
        context = '\n\n'.join(retrieved)
        pred = generate_answer(context, budget_qa[i]['question'])
        f1 = compute_f1(pred, budget_qa[i]['answer'])
        f1_scores.append(f1)
        storage_saved.append(1.0 - len(selected) / len(chunks))

    mean_f1 = np.mean(f1_scores)
    mean_savings = np.mean(storage_saved) * 100

    budget_results[ratio] = {
        'f1_mean': float(mean_f1),
        'storage_savings_pct': float(mean_savings),
    }
    print(f"  Budget {int(ratio*100)}%: F1={mean_f1:.4f}, Savings={mean_savings:.1f}%")

    # Checkpoint
    with open(f"{PROJECT_DIR}/results/budget_sensitivity.json", 'w') as f:
        json.dump({
            'baseline_f1': float(budget_baseline_f1),
            'results': {str(k): v for k, v in budget_results.items()},
            'n_examples': n_budget,
        }, f, indent=2)

print("\nBudget sensitivity complete!")

In [ ]:
# Print the table for the paper
print("="*60)
print("TABLE 5 UPDATE: Budget Sensitivity")
print("="*60)
print(f"{'Budget Ratio':<15} {'F1 Score':<12} {'Storage Savings':<15}")
print("-"*42)
for ratio in BUDGET_RATIOS:
    r = budget_results[ratio]
    star = " ***" if ratio == 0.30 else ""
    print(f"{int(ratio*100)}%{'':<12} {r['f1_mean']:<12.4f} {r['storage_savings_pct']:<15.1f}{star}")
print(f"\nBaseline (100%): {budget_baseline_f1:.4f}")

---
## Summary

In [ ]:
print("="*60)
print("FINAL POLISH RESULTS")
print("="*60)

print(f"\n--- 3-Seed Medium Docs ---")
print(f"Baseline: {np.mean(baseline_per_seed):.4f} +/- {np.std(baseline_per_seed):.4f}")
print(f"BudgetMem: {np.mean(budgetmem_per_seed):.4f} +/- {np.std(budgetmem_per_seed):.4f}")

print(f"\n--- Budget Sensitivity ---")
for ratio in BUDGET_RATIOS:
    r = budget_results[ratio]
    print(f"  {int(ratio*100)}%: F1={r['f1_mean']:.4f}  savings={r['storage_savings_pct']:.1f}%")

# Save combined results
all_final = {
    'three_seed': three_seed_results,
    'budget_sensitivity': {
        'baseline_f1': float(budget_baseline_f1),
        'results': {str(k): v for k, v in budget_results.items()},
        'n_examples': n_budget,
    },
    'timestamp': datetime.now().isoformat()
}
with open(f"{PROJECT_DIR}/results/FINAL_POLISH_RESULTS.json", 'w') as f:
    json.dump(all_final, f, indent=2)

print(f"\nAll results saved to {PROJECT_DIR}/results/FINAL_POLISH_RESULTS.json")
print("Download this file and share it back to update the paper.")